<a href="https://colab.research.google.com/github/zayfer-ux/inventario-futbol-turso/blob/main/Inventario_futbol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALAR EL CONECTOR

Google Colab necesita instalar el paquete que permite comunicarse con la nueva base de datos Turso.

In [ ]:
!pip install -q turso_serverless requests

CONECTAR GOOGLE COLAB CON TURSO

In [ ]:
import turso_serverless
from getpass import getpass

url = "libsql://inventariofutbol-misaelzy.aws-us-west-2.turso.io"
token = getpass("Ingresa tu token de Turso: ").strip()

conexion = turso_serverless.connect(url, auth_token=token)
print("Conexión exitosa")

#CREAR TABLA

In [ ]:
conexion.execute("""
CREATE TABLE IF NOT EXISTS equipos (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    cantidad INTEGER NOT NULL,
    estado TEXT NOT NULL
)
""")

conexion.commit()
print("Tabla lista")

CRUD

CREATE: Agregar un equipo y usar Random Word API

In [ ]:
import requests
import random

endpoint = "https://random-word-api.herokuapp.com/word?number=1&lang=es"
nombre = requests.get(endpoint).json()[0]
cantidad = random.randint(1, 20)
estado = random.choice(["Bueno", "Regular", "Nuevo"])

conexion.execute(
    "INSERT INTO equipos (nombre, cantidad, estado) VALUES (?, ?, ?)",
    (nombre, cantidad, estado)
)

conexion.commit()
print(nombre, cantidad, estado)


#Probar endpoint

In [ ]:
import requests

endpoint = "https://random-word-api.herokuapp.com/word?number=1&lang=es"
respuesta = requests.get(endpoint)

print(respuesta.json())

READ: Mostrar el inventario

In [ ]:
datos = conexion.execute("SELECT * FROM equipos").fetchall()

for dato in datos:
    print(dato)

UPDATE: Actualizar un equipo

In [ ]:
id_equipo = int(input("ID del equipo: "))
cantidad = int(input("Nueva cantidad: "))
estado = input("Nuevo estado: ")

conexion.execute(
    "UPDATE equipos SET cantidad = ?, estado = ? WHERE id = ?",
    (cantidad, estado, id_equipo)
)

conexion.commit()
print("Equipo actualizado")

DELETE: Eliminar un equipo

In [ ]:
id_equipo = int(input("ID que deseas eliminar: "))

conexion.execute(
    "DELETE FROM equipos WHERE id = ?",
    (id_equipo,)
)

conexion.commit()
print("Equipo eliminado")

#Datos generales del encargado

In [ ]:
# CREAR LA NUEVA TABLA DE PRÉSTAMOS
conexion.execute("""
CREATE TABLE IF NOT EXISTS prestamos (
    id INTEGER PRIMARY KEY,
    equipo_id INTEGER NOT NULL,
    responsable TEXT NOT NULL,
    cantidad_prestada INTEGER NOT NULL,
    fecha_salida TEXT NOT NULL,
    estado_prestamo TEXT NOT NULL,
    FOREIGN KEY (equipo_id) REFERENCES equipos(id)
)
""")

# Hacer commit para guardar la tabla
conexion.commit()
print("Tabla 'prestamos' lista")

# INSERTAR UN REGISTRO DE PRUEBA
# Usamos el id=1 (que según tu primera imagen corresponde a "Balones")
conexion.execute("""
INSERT INTO prestamos (equipo_id, responsable, cantidad_prestada, fecha_salida, estado_prestamo)
VALUES (1, 'Carlos Entrenador', 5, '2026-09-21', 'Activo')
""")

# Hacer commit para guardar el registro
conexion.commit()
print("Registro de prueba añadido a 'prestamos'")